# Naive Bayes

**Topic:** Supervised Learning — Classification

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from ipywidgets import Dropdown, FloatSlider, IntSlider, Output, HBox, VBox
from IPython.display import display, clear_output
from scipy import stats
from sklearn.datasets import load_breast_cancer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, log_loss, roc_auc_score
from tkh_utils import PALETTE, base_layout

np.random.seed(42)


---
## What you'll explore

By the end of this demo you will be able to:

- **Describe** how Naive Bayes multiplies a prior by each feature's likelihood and normalizes the result to get a probability for each class
- **Interpret** why a single unseen feature-class combination forces a probability all the way to zero, and how smoothing fixes it
- **Explain** why the independence assumption is almost always wrong, and what that costs you

> **Tip:** In the first widget, set Outlook to "Overcast" and watch Score(No) collapse to zero. In the second widget, raise α from 0 and watch Score(No) come back to life, then keep raising it and watch every bar drift toward the dashed uniform line. In the third widget, drag from 0 to 5 and watch accuracy stay flat while the model's confidence balloons.

---
## How we got here

Naive Bayes is a direct application of probability theory to classification:

- **[math/probability_and_combinatorics/04_bayes_theorem_deep_dive.ipynb](../math/probability_and_combinatorics/04_bayes_theorem_deep_dive.ipynb)** — Bayes theorem is the entire mathematical foundation of this algorithm; everything here is an application of that single formula
- **[statistics/02_conditional_probability_independence.ipynb](../statistics/02_conditional_probability_independence.ipynb)** — the naive assumption is conditional independence: $P(x_1, x_2 \mid C) = P(x_1 \mid C) \cdot P(x_2 \mid C)$; this is the simplification that makes the algorithm tractable

---
## Why this matters for data science

Naive Bayes is the canonical example of a probabilistic classifier. Its "naive" assumption — that features are conditionally independent given the class — makes the math extremely tractable and the model extremely fast. Despite the assumption being almost always wrong in practice, it often produces competitive classification results.

For text classification (spam filtering, sentiment analysis, topic classification), Naive Bayes is still competitive with much more complex models. Picking the class with the highest score can be right even when the scores themselves are badly wrong, because classification only needs the *ranking* of the two classes to come out correct, not the exact numbers. The third widget below shows this happening directly.

---
## Where it sits on the spectrum

See **[ml_concepts/13_interpretability_vs_complexity.ipynb](../ml_concepts/13_interpretability_vs_complexity.ipynb)** for the full spectrum.

Naive Bayes sits in the **upper-left portion** of the spectrum: medium-high interpretability, very low complexity. You can inspect the learned prior and class-conditional distributions directly and understand exactly what the model has learned about each feature.

It is the fastest algorithm in supervised learning by a large margin. Training requires only computing mean and variance for each feature per class (for GaussianNB) — a single pass through the data.

---
## How it learns

Think about how a spam filter works. You have thousands of emails labeled spam or not-spam. For each word, you ask: how often does this word appear in spam emails? How often in legitimate ones? If "Nigerian prince" appears 95% of the time in spam and 0.2% of the time in legitimate email, that word is strong evidence of spam.

Naive Bayes formalizes this idea using Bayes theorem. For each new email, you multiply the probabilities of each word appearing given each class, weight by the prior probability of that class (the base rate of spam in your inbox), and predict the class with the highest product.

The "naive" part is the independence assumption: you assume that knowing whether "prince" appears tells you nothing additional about whether "Nigerian" appears, given you already know it is spam. This is obviously false — they tend to appear together. But the model still works surprisingly well because the errors from this assumption often cancel out across many features.

---
## The math behind it

**Bayes theorem** applied to classification:

$$P(C \mid \mathbf{x}) = \frac{P(\mathbf{x} \mid C) \cdot P(C)}{P(\mathbf{x})}$$

- $P(C)$ — prior probability of class $C$ (estimated from training label frequencies)
- $P(\mathbf{x} \mid C)$ — likelihood of observing features $\mathbf{x}$ given class $C$
- $P(\mathbf{x})$ — evidence (constant across classes; dropped in practice)

**Naive independence assumption:**

$$P(\mathbf{x} \mid C) = \prod_{j=1}^{p} P(x_j \mid C)$$

**Prediction rule** (log-space to avoid numerical underflow):

$$\hat{y} = \underset{C}{\arg\max} \left[ \log P(C) + \sum_{j=1}^{p} \log P(x_j \mid C) \right]$$

**Gaussian Naive Bayes** (for continuous features) models $P(x_j \mid C)$ as:

$$P(x_j \mid C) = \frac{1}{\sqrt{2\pi\sigma_{jC}^2}} \exp\!\left(-\frac{(x_j - \mu_{jC})^2}{2\sigma_{jC}^2}\right)$$

where $\mu_{jC}$ and $\sigma_{jC}^2$ are the mean and variance of feature $j$ estimated from class $C$'s training examples.

---
## Try it yourself

In [ ]:
_tennis_df = pd.DataFrame([
    {"Outlook": "Sunny",    "Humidity": "High",   "Windy": "False", "Play": "No"},
    {"Outlook": "Sunny",    "Humidity": "Normal", "Windy": "True",  "Play": "Yes"},
    {"Outlook": "Overcast", "Humidity": "High",   "Windy": "False", "Play": "Yes"},
    {"Outlook": "Rain",     "Humidity": "Normal", "Windy": "False", "Play": "Yes"},
    {"Outlook": "Rain",     "Humidity": "Normal", "Windy": "True",  "Play": "No"},
    {"Outlook": "Overcast", "Humidity": "Low",    "Windy": "True",  "Play": "Yes"},
    {"Outlook": "Sunny",    "Humidity": "High",   "Windy": "True",  "Play": "No"},
    {"Outlook": "Rain",     "Humidity": "Low",    "Windy": "True",  "Play": "Yes"},
])
_OUTLOOK_CATS = ["Sunny", "Overcast", "Rain"]
_HUMIDITY_CATS = ["High", "Normal", "Low"]
_WINDY_CATS = ["False", "True"]

print("The 8-row training set used by both widgets below:")
display(_tennis_df)

In [ ]:
_w1_out = Output()
_w1_caption = widgets.HTML()

_w1_outlook_dd = Dropdown(
    options=_OUTLOOK_CATS, value="Rain",
    description="Outlook:", style={"description_width": "90px"},
    layout=widgets.Layout(width="260px"),
)
_w1_humidity_dd = Dropdown(
    options=_HUMIDITY_CATS, value="High",
    description="Humidity:", style={"description_width": "90px"},
    layout=widgets.Layout(width="260px"),
)
_w1_windy_dd = Dropdown(
    options=_WINDY_CATS, value="False",
    description="Windy:", style={"description_width": "90px"},
    layout=widgets.Layout(width="260px"),
)

def _w1_cond_prob(col, val, play_val, n_class):
    match = ((_tennis_df[col] == val) & (_tennis_df["Play"] == play_val)).sum()
    return match / n_class

def _w1_zero_culprits(outlook, humidity, windy, play_val):
    culprits = []
    for col, val in [("Outlook", outlook), ("Humidity", humidity), ("Windy", windy)]:
        count = ((_tennis_df[col] == val) & (_tennis_df["Play"] == play_val)).sum()
        if count == 0:
            culprits.append(f"{col}={val}")
    return culprits

def _w1_render(change=None):
    outlook = _w1_outlook_dd.value
    humidity = _w1_humidity_dd.value
    windy = _w1_windy_dd.value

    n_total = len(_tennis_df)
    n_yes = (_tennis_df["Play"] == "Yes").sum()
    n_no = (_tennis_df["Play"] == "No").sum()
    p_yes, p_no = n_yes / n_total, n_no / n_total

    p_outlook_yes = _w1_cond_prob("Outlook", outlook, "Yes", n_yes)
    p_outlook_no = _w1_cond_prob("Outlook", outlook, "No", n_no)
    p_humidity_yes = _w1_cond_prob("Humidity", humidity, "Yes", n_yes)
    p_humidity_no = _w1_cond_prob("Humidity", humidity, "No", n_no)
    p_windy_yes = _w1_cond_prob("Windy", windy, "Yes", n_yes)
    p_windy_no = _w1_cond_prob("Windy", windy, "No", n_no)

    score_yes = p_yes * p_outlook_yes * p_humidity_yes * p_windy_yes
    score_no = p_no * p_outlook_no * p_humidity_no * p_windy_no
    total = score_yes + score_no
    post_yes = score_yes / total if total > 0 else 0.5
    post_no = score_no / total if total > 0 else 0.5

    factor_labels = ["Prior<br>P(Play)", f"P(Outlook=<br>{outlook}|Play)",
                      f"P(Humidity=<br>{humidity}|Play)", f"P(Windy=<br>{windy}|Play)"]
    yes_factors = [p_yes, p_outlook_yes, p_humidity_yes, p_windy_yes]
    no_factors = [p_no, p_outlook_no, p_humidity_no, p_windy_no]

    fig = make_subplots(
        rows=1, cols=3,
        subplot_titles=("Factors", "Score", "Posterior"),
        column_widths=[0.5, 0.25, 0.25],
    )
    fig.add_trace(go.Bar(
        x=factor_labels, y=yes_factors, name="Play = Yes",
        marker_color=PALETTE["secondary"],
        text=[f"{v:.3f}" for v in yes_factors], textposition="auto",
    ), row=1, col=1)
    fig.add_trace(go.Bar(
        x=factor_labels, y=no_factors, name="Play = No",
        marker_color=PALETTE["primary"],
        text=[f"{v:.3f}" for v in no_factors], textposition="auto",
    ), row=1, col=1)
    fig.add_trace(go.Bar(
        x=["Score(Yes)", "Score(No)"], y=[score_yes, score_no],
        marker_color=[PALETTE["secondary"], PALETTE["primary"]],
        text=[f"{score_yes:.4f}", f"{score_no:.4f}"],
        textposition="auto", showlegend=False,
    ), row=1, col=2)
    fig.add_trace(go.Bar(
        x=["P(Yes|X)", "P(No|X)"], y=[post_yes, post_no],
        marker_color=[PALETTE["secondary"], PALETTE["primary"]],
        text=[f"{post_yes:.1%}", f"{post_no:.1%}"],
        textposition="auto", showlegend=False,
    ), row=1, col=3)

    predicted = "Yes" if post_yes >= post_no else "No"
    fig.update_layout(**base_layout(title="Bayes Calculator: Factors → Score → Posterior").to_plotly_json())
    fig.update_layout(height=440, barmode="group")

    with _w1_out:
        clear_output(wait=True)
        display(go.FigureWidget(fig))

    zero_no = _w1_zero_culprits(outlook, humidity, windy, "No")
    zero_yes = _w1_zero_culprits(outlook, humidity, windy, "Yes")
    if zero_no:
        combo_desc = " and ".join(zero_no)
        zero_note = (
            f" The combination {combo_desc} never appears in a Play=No row in the 8 "
            f"training rows, so its probability is zero and it drags the whole product to "
            f"zero. The next widget fixes this."
        )
    elif zero_yes:
        combo_desc = " and ".join(zero_yes)
        zero_note = (
            f" The combination {combo_desc} never appears in a Play=Yes row in the 8 "
            f"training rows, so its probability is zero and it drags the whole product to "
            f"zero. The next widget fixes this."
        )
    else:
        zero_note = ""

    _w1_caption.value = (
        f"<b>Outlook={outlook}, Humidity={humidity}, Windy={windy}</b> → prior × likelihoods "
        f"give Score(Yes)={score_yes:.4f} and Score(No)={score_no:.4f}. Dividing each by "
        f"their sum (the Bayesian normalization step) predicts <b>Play = {predicted}</b> "
        f"with {max(post_yes, post_no):.1%} confidence.{zero_note}"
    )

for _ctrl in (_w1_outlook_dd, _w1_humidity_dd, _w1_windy_dd):
    _ctrl.observe(_w1_render, names="value")

display(VBox([HBox([_w1_outlook_dd, _w1_humidity_dd, _w1_windy_dd]), _w1_out, _w1_caption]))

if not globals().get("_nb12_w1_initialized", False):
    _w1_render()
    _nb12_w1_initialized = True

---
## What's happening: multiply, then normalize

The 8-row training set above is small enough to hand-check. For a chosen combination of Outlook, Humidity, and Windy, the calculator multiplies the prior by each feature's likelihood to get an unnormalized "score" for each class, then divides each score by their sum to get the final posterior probability.

That division is the Bayesian part. It is exactly Bayes theorem, with the denominator (the "evidence") computed implicitly as whatever normalizes the two scores to sum to 1, instead of being calculated directly.

Ten of the eighteen possible Outlook/Humidity/Windy combinations force one score to exactly zero — you may have already landed on one above. That happens because the missing combination was never seen with that class anywhere in the 8 rows, and multiplying by zero collapses the entire product to zero, no matter how strong the other evidence is. The model becomes absurdly overconfident, predicting the other class with 100% certainty from a single missing combination in a tiny dataset. The next widget fixes this.

In [ ]:
_w2_out = Output()
_w2_caption = widgets.HTML()

_w2_outlook_dd = Dropdown(
    options=_OUTLOOK_CATS, value="Overcast",
    description="Outlook:", style={"description_width": "90px"},
    layout=widgets.Layout(width="260px"),
)
_w2_humidity_dd = Dropdown(
    options=_HUMIDITY_CATS, value="High",
    description="Humidity:", style={"description_width": "90px"},
    layout=widgets.Layout(width="260px"),
)
_w2_windy_dd = Dropdown(
    options=_WINDY_CATS, value="False",
    description="Windy:", style={"description_width": "90px"},
    layout=widgets.Layout(width="260px"),
)
_w2_alpha_slider = FloatSlider(
    value=0.0, min=0.0, max=20.0, step=0.5,
    description="Smoothing α:",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="420px"),
)

def _w2_smoothed_prob(col, val, play_val, n_class, alpha, k_categories):
    match = ((_tennis_df[col] == val) & (_tennis_df["Play"] == play_val)).sum()
    return (match + alpha) / (n_class + alpha * k_categories)

def _w2_zero_culprits(outlook, humidity, windy, play_val):
    culprits = []
    for col, val in [("Outlook", outlook), ("Humidity", humidity), ("Windy", windy)]:
        count = ((_tennis_df[col] == val) & (_tennis_df["Play"] == play_val)).sum()
        if count == 0:
            culprits.append(f"{col}={val}")
    return culprits

def _w2_render(change=None):
    outlook = _w2_outlook_dd.value
    humidity = _w2_humidity_dd.value
    windy = _w2_windy_dd.value
    alpha = _w2_alpha_slider.value

    n_yes = (_tennis_df["Play"] == "Yes").sum()
    n_no = (_tennis_df["Play"] == "No").sum()
    p_yes, p_no = n_yes / len(_tennis_df), n_no / len(_tennis_df)

    outlook_yes = [_w2_smoothed_prob("Outlook", c, "Yes", n_yes, alpha, 3) for c in _OUTLOOK_CATS]
    outlook_no = [_w2_smoothed_prob("Outlook", c, "No", n_no, alpha, 3) for c in _OUTLOOK_CATS]

    p_o_yes = _w2_smoothed_prob("Outlook", outlook, "Yes", n_yes, alpha, 3)
    p_o_no = _w2_smoothed_prob("Outlook", outlook, "No", n_no, alpha, 3)
    p_h_yes = _w2_smoothed_prob("Humidity", humidity, "Yes", n_yes, alpha, 3)
    p_h_no = _w2_smoothed_prob("Humidity", humidity, "No", n_no, alpha, 3)
    p_w_yes = _w2_smoothed_prob("Windy", windy, "Yes", n_yes, alpha, 2)
    p_w_no = _w2_smoothed_prob("Windy", windy, "No", n_no, alpha, 2)
    score_yes = p_yes * p_o_yes * p_h_yes * p_w_yes
    score_no = p_no * p_o_no * p_h_no * p_w_no

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("P(Outlook | Play)", "Score(Yes) vs Score(No)"),
    )
    fig.add_trace(go.Bar(x=_OUTLOOK_CATS, y=outlook_yes, name="Play = Yes",
                          marker_color=PALETTE["secondary"]), row=1, col=1)
    fig.add_trace(go.Bar(x=_OUTLOOK_CATS, y=outlook_no, name="Play = No",
                          marker_color=PALETTE["primary"]), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=_OUTLOOK_CATS, y=[1 / 3] * 3, mode="lines",
        line=dict(color=PALETTE["muted"], dash="dot"),
        name="Uniform (1/3)",
    ), row=1, col=1)
    fig.add_trace(go.Bar(
        x=["Score(Yes)", "Score(No)"], y=[score_yes, score_no],
        marker_color=[PALETTE["secondary"], PALETTE["primary"]],
        text=[f"{score_yes:.4f}", f"{score_no:.4f}"], textposition="auto",
        showlegend=False,
    ), row=1, col=2)

    fig.update_layout(**base_layout(title="Laplace Smoothing").to_plotly_json())
    fig.update_layout(height=420, barmode="group", yaxis1=dict(range=[0, 0.8]))

    with _w2_out:
        clear_output(wait=True)
        display(go.FigureWidget(fig))

    zero_no = _w2_zero_culprits(outlook, humidity, windy, "No")
    zero_yes = _w2_zero_culprits(outlook, humidity, windy, "Yes")
    if alpha == 0 and (zero_no or zero_yes):
        zero_class = "No" if zero_no else "Yes"
        culprits = zero_no if zero_no else zero_yes
        combo_desc = " and ".join(culprits)
        note_extra = (
            " Note: the left chart only plots Outlook, so since this zero comes from "
            "Humidity, you'll see it in this caption and the Score panel but not in the "
            "bars on the left."
            if any(c.startswith("Humidity") for c in culprits)
            and not any(c.startswith("Outlook") for c in culprits)
            else ""
        )
        _w2_caption.value = (
            f"<b>α=0 (no smoothing):</b> Score({zero_class}) is exactly <b>0</b> — the "
            f"combination {combo_desc} never appears in a Play={zero_class} row anywhere in "
            f"the 8-row training set, so that one gap forces total certainty in the other "
            f"class. Raise α above 0 to fix this.{note_extra}"
        )
    else:
        _w2_caption.value = (
            f"<b>α={alpha:g}:</b> Score(Yes)={score_yes:.4f}, Score(No)={score_no:.4f} — "
            f"both are nonzero now. As α grows, every bar on the left keeps sliding toward "
            f"the dashed uniform line (1/3): at very high α the model stops trusting the "
            f"training data at all and just predicts the uniform rate for every category."
        )

for _ctrl in (_w2_outlook_dd, _w2_humidity_dd, _w2_windy_dd, _w2_alpha_slider):
    _ctrl.observe(_w2_render, names="value")

display(VBox([HBox([_w2_outlook_dd, _w2_humidity_dd, _w2_windy_dd]), _w2_alpha_slider,
              _w2_out, _w2_caption]))

if not globals().get("_nb12_w2_initialized", False):
    _w2_render()
    _nb12_w2_initialized = True

---
## What's happening: zeros and smoothing

Smoothing adds a small constant α to every count before dividing, so no observed-zero count ever produces a probability of exactly zero. Raise α from 0 in the widget above and watch the zeroed score become nonzero immediately. Keep raising α, and every bar in the left-hand chart drifts toward the dashed line at 1/3 — the uniform probability across Outlook's three categories:

| α | P(Sunny\|No) | P(Overcast\|No) | P(Rain\|No) |
|---|---|---|---|
| 0 | 0.667 | 0.000 | 0.333 |
| 0.5 | 0.556 | 0.111 | 0.333 |
| 1 | 0.500 | 0.167 | 0.333 |
| 5 | 0.389 | 0.278 | 0.333 |
| 20 | 0.349 | 0.317 | 0.333 |

That's the tradeoff: enough smoothing prevents zero-probability overconfidence, but too much smoothing drowns out real patterns in the data and pushes every prediction toward "no information at all," regardless of what the training set actually shows.

Note that the left chart plots Outlook only. If you zero out the score by picking a Humidity=Low combination instead, you'll see it in the caption and the Score panel, but the left-hand bars — which only track Outlook — will show nothing unusual.

In [ ]:
_w3_out = Output()
_w3_caption = widgets.HTML()

_bc = load_breast_cancer(as_frame=True)
_w3_X, _w3_y = _bc.data, _bc.target
_w3_Xtr, _w3_Xte, _w3_ytr, _w3_yte = train_test_split(
    _w3_X, _w3_y, test_size=0.2, random_state=42, stratify=_w3_y
)
_w3_BASE = ["mean radius", "mean texture", "mean smoothness"]
_w3_REDUNDANT = ["mean perimeter", "mean area", "worst radius", "worst perimeter", "worst area"]

_w3_k_values = list(range(6))
_w3_accuracy = []
_w3_auc = []
_w3_logloss = []
_w3_confidence = []
for _k in _w3_k_values:
    _feats = _w3_BASE + _w3_REDUNDANT[:_k]
    _w3_model = GaussianNB()
    _w3_model.fit(_w3_Xtr[_feats], _w3_ytr)
    _proba = _w3_model.predict_proba(_w3_Xte[_feats])
    _pred = _w3_model.predict(_w3_Xte[_feats])
    _w3_accuracy.append(accuracy_score(_w3_yte, _pred))
    _w3_auc.append(roc_auc_score(_w3_yte, _proba[:, 1]))
    _w3_logloss.append(log_loss(_w3_yte, _proba))
    _w3_confidence.append(_proba.max(axis=1))

_w3_slider = IntSlider(
    value=0, min=0, max=5, step=1,
    description="Extra size measurements:",
    style={"description_width": "160px"},
    layout=widgets.Layout(width="420px"),
)

def _w3_render(change=None):
    k = _w3_slider.value
    conf = _w3_confidence[k]
    pct99 = (conf > 0.99).mean() * 100
    pct99_base = (_w3_confidence[0] > 0.99).mean() * 100

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("Accuracy across all six models", "Confidence distribution"),
    )
    fig.add_trace(go.Scatter(
        x=_w3_k_values, y=_w3_accuracy, mode="lines+markers",
        line=dict(color=PALETTE["primary"], width=2.5), marker=dict(size=7),
        name="Accuracy",
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=_w3_k_values, y=_w3_auc, mode="lines+markers",
        line=dict(color=PALETTE["muted"], width=2, dash="dot"), marker=dict(size=6),
        name="AUC",
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=[k], y=[_w3_accuracy[k]], mode="markers",
        marker=dict(size=16, color=PALETTE["accent"], symbol="circle-open", line=dict(width=3)),
        name=f"k={k}",
    ), row=1, col=1)
    fig.add_trace(go.Histogram(
        x=conf, xbins=dict(start=0.5, end=1.0, size=0.025),
        marker_color=PALETTE["secondary"], name="Confidence", showlegend=False,
    ), row=1, col=2)

    fig.update_layout(**base_layout(title="Redundant Features and Overconfidence").to_plotly_json())
    fig.update_layout(height=420, showlegend=False)
    fig.update_xaxes(title_text="Extra size measurements (k)", row=1, col=1)
    fig.update_yaxes(title_text="Score", range=[0.5, 1.0], row=1, col=1)
    fig.update_xaxes(title_text="Max posterior probability", range=[0.5, 1.0], row=1, col=2)
    fig.update_yaxes(title_text="Count", row=1, col=2)

    with _w3_out:
        clear_output(wait=True)
        display(go.FigureWidget(fig))

    _w3_caption.value = (
        f"<b>k={k} extra measurement(s) added</b> on top of the 3 base features: accuracy is "
        f"{_w3_accuracy[k]:.1%} (log loss {_w3_logloss[k]:.3f}), close to k=0's "
        f"{_w3_accuracy[0]:.1%} (log loss {_w3_logloss[0]:.3f}) — but {pct99:.1f}% of test "
        f"predictions are now above 99% confidence, versus {pct99_base:.1f}% at k=0. The "
        f"extra features are near-duplicates of ones already in the model, so Naive Bayes "
        f"double- and triple-counts the same evidence and becomes falsely certain without "
        f"becoming more accurate."
    )

_w3_slider.observe(_w3_render, names="value")

display(VBox([_w3_slider, _w3_out, _w3_caption]))

if not globals().get("_nb12_w3_initialized", False):
    _w3_render()
    _nb12_w3_initialized = True

---
## What's happening: the assumption's real cost

The three "base" features above (mean radius, mean texture, mean smoothness) are joined one at a time by mean perimeter, mean area, worst radius, worst perimeter, and worst area. The first three of those are 0.987 to 0.998 correlated with mean radius, because radius, perimeter, and area are three different ways of measuring the same circle.

Naive Bayes assumes every feature is independent evidence. It has no way to know that "mean radius," "mean perimeter," and "mean area" are all telling it the same thing about the same tumor, so as you add them it counts that one piece of evidence two and three times over.

Watch the left panel as you drag the slider: accuracy does not hold steady. It wanders between about 86% and 91% with no clear trend — adding redundant features does not reliably help or hurt how many predictions are correct.

The right panel tells a different story. At k=0, the model is more than 99% confident in about 29% of its predictions. By k=5, that's grown to about 89% of predictions — on the same test set, with no real gain in accuracy. Log loss, which penalizes overconfident wrong answers heavily, more than doubles over the same range (0.210 to 0.543).

That's the real cost of the independence assumption: it mostly damages *how confident the model claims to be*, not *which class it picks*. That's why Naive Bayes is a reliable classifier and an unreliable probability estimate — and why the weaknesses table below lists poor calibration.

---
## Key hyperparameters

**`var_smoothing`** (GaussianNB, default `1e-9`) — adds a fraction of the largest variance to all variances for numerical stability. Increase this if you get numerical warnings.

**`priors`** (default `None`, estimated from data) — class prior probabilities. Set this manually if your training data does not reflect real-world class frequencies (imbalanced training sets).

**`alpha`** (MultinomialNB, default `1.0`) — Laplace/Lidstone smoothing to prevent zero probabilities for unseen feature-class combinations, exactly like the α slider above. The default of 1.0 is standard Laplace smoothing and is usually left alone for text.

For the full list of hyperparameters, see the sklearn documentation:
[https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.GaussianNB.html](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.GaussianNB.html)

---
## Strengths and weaknesses

| Strengths | Weaknesses |
|-----------|------------|
| Extremely fast training and prediction — single pass through data | Naive independence assumption is almost always violated, and correlated features get counted multiple times as evidence |
| Works well with very small training sets | Poor probability calibration (posteriors are often extreme: near 0 or 1) |
| Handles high-dimensional data efficiently | Cannot model feature interactions |
| Naturally handles multiclass without any extra configuration | Continuous features require assuming a distribution (Gaussian, etc.) |
| Incremental learning: can update with new data without full retraining | Fits a Gaussian to every continuous feature whether or not it is actually bell-shaped — the `area error` example later in this notebook shows this concretely |

---
## When to use it / When NOT to use it

| Use it when | Do NOT use it when |
|---|---|
| Text classification: spam, sentiment, topic labeling | Features are strongly correlated with each other |
| Extremely fast training with limited compute or memory | You need well-calibrated probability outputs |
| Very small training sets where more complex models overfit | The independence assumption is clearly violated |
| Online/streaming learning where you update the model incrementally | You need to model interactions between features |
| Baseline that is hard to beat on text data | The relationship between features and classes is complex and nonlinear |

---
## Real-world example: Predicting malignant vs. benign tumors

The Breast Cancer Wisconsin dataset has 569 tumor samples, each with 30 continuous measurements (radius, texture, smoothness, and more), labeled malignant or benign. It's a good match for Gaussian Naive Bayes: every feature is a continuous measurement, so fitting a bell curve to each one is at least a reasonable starting assumption — unlike a dataset full of yes/no or category columns, where a different Naive Bayes variant would be the better fit.

In [ ]:
_bc = load_breast_cancer(as_frame=True)
_X, _y = _bc.data, _bc.target
_Xtr, _Xte, _ytr, _yte = train_test_split(
    _X, _y, test_size=0.2, random_state=42, stratify=_y
)

_model = GaussianNB()
_model.fit(_Xtr, _ytr)
_pred = _model.predict(_Xte)
_prob = _model.predict_proba(_Xte)[:, 1]

print(classification_report(_yte, _pred, target_names=_bc.target_names))
print(f"ROC-AUC: {roc_auc_score(_yte, _prob):.4f}")
print()
print("Class priors learned from training data:")
for _c, _p in zip(_model.classes_, _model.class_prior_):
    print(f"  {_bc.target_names[_c]}: {_p:.4f}")

---
## Results: what 93.9% actually means here

Naive Bayes reaches 93.9% accuracy and 0.988 ROC-AUC on tumors it never saw during training. For context, it gets there by fitting two numbers per feature per class — a mean and a spread — for 30 features across 2 classes: 120 numbers total, learned in a single pass over the data. A random forest solving the same problem builds dozens of trees, each with its own set of splits, and takes considerably longer to both train and explain.

Look at recall specifically, not just the headline accuracy: the model misses about 10% of actual malignant cases (recall 0.905, 4 out of 42 in this test set). In a screening context, that miss rate — not overall accuracy — is the number a clinician actually cares about, since a missed malignant tumor is a much costlier mistake than a false alarm on a benign one.

The learned priors, about 37% malignant and 63% benign, are just the training label frequencies — the same `priors` hyperparameter mentioned above. If real-world prevalence looked different from this training set, that's exactly the parameter you would override.

In [ ]:
_selected = [
    "worst perimeter",
    "worst concave points",
    "mean compactness",
    "mean texture",
    "mean fractal dimension",
    "area error",
]

_dist_model = GaussianNB()
_dist_model.fit(_X, _y)
_feature_names = list(_X.columns)

_dist_fig = make_subplots(rows=2, cols=3, subplot_titles=_selected)
_class_names = list(_bc.target_names)
_class_colors = [PALETTE["primary"], PALETTE["secondary"]]

for _idx, _feat in enumerate(_selected):
    _row, _col = _idx // 3 + 1, _idx % 3 + 1
    _fi = _feature_names.index(_feat)
    for _ci, (_cname, _color) in enumerate(zip(_class_names, _class_colors)):
        _vals = _X.loc[_y == _ci, _feat]
        _dist_fig.add_trace(go.Histogram(
            x=_vals, histnorm="probability density",
            marker_color=_color, opacity=0.45,
            name=_cname, legendgroup=_cname, showlegend=(_idx == 0),
        ), row=_row, col=_col)
        _mu = _dist_model.theta_[_ci, _fi]
        _sigma = np.sqrt(_dist_model.var_[_ci, _fi])
        _x_range = np.linspace(_vals.min(), _vals.max(), 200)
        _dist_fig.add_trace(go.Scatter(
            x=_x_range, y=stats.norm.pdf(_x_range, _mu, _sigma),
            mode="lines",
            line=dict(color=_color, width=2, dash="solid" if _ci == 0 else "dash"),
            name=f"{_cname} fit", legendgroup=_cname, showlegend=False,
        ), row=_row, col=_col)

_dist_fig.update_layout(**base_layout(title="Class-Conditional Feature Distributions").to_plotly_json())
_dist_fig.update_layout(height=620, barmode="overlay")
_dist_fig.show()

---
## What the plot shows

Each panel plots the actual distribution of one feature for malignant and benign tumors, with the Gaussian curve Naive Bayes fits on top. This fit uses all 569 rows, not just the training split above — here we're inspecting what the model learned, not measuring how well it generalizes.

- **Notice:** `worst perimeter` (top-left) has the least overlap of any of the 30 features in this dataset. Malignant tumors average 141 units against 87 for benign — the two curves barely touch. That gap is what the model is using to separate the classes.
- **Notice:** `worst concave points` and `mean compactness` show progressively more overlap — still useful, but less decisive on their own.
- **Notice:** `mean texture` overlaps heavily, and `mean fractal dimension` (bottom-middle) shows almost no separation at all: malignant tumors average 0.06268, benign average 0.06287. The two curves are effectively the same curve — this feature contributes almost nothing, and the model has no way to know that.
- **Notice:** `area error` (bottom-right) is one of the most separated features in the whole dataset, but its actual values are heavily right-skewed — most tumors cluster at low values with a long tail of large ones. The fitted bell curve doesn't know that: it puts about 11.8% of its probability mass below zero, on an area measurement that cannot be negative. The smallest `area error` actually recorded for a malignant tumor is 14.0 — nowhere close to the curve's lower tail.
- **Notice:** And yet `area error` still works. It's one of the six most separated features here, despite the model's assumption about its shape being visibly wrong. The ranking of malignant vs. benign survives even when the underlying probability model does not — the same effect from the third widget above, now visible in real data.

> **Discussion question:** The bell curve the model fits to `area error` puts almost 12% of its probability on negative values, which are impossible. Yet this is one of the most useful features in the dataset. Why does the model still work, and what would you do to this feature before feeding it in?

---
## Real-world example: Spam filtering with word counts

Four earlier sections in this notebook use spam filtering as the motivating example. Here's the actual thing: a tiny spam filter built on word counts, using a different Naive Bayes variant. `GaussianNB` fits bell curves to continuous measurements; text is made of discrete word counts, so this section switches to `MultinomialNB`, which counts how often each word appears under each class instead.

In [ ]:
_msgs = [
    ("free money click now", "spam"),
    ("win free prize click here", "spam"),
    ("claim your free money today", "spam"),
    ("urgent click to claim prize", "spam"),
    ("free trial click now urgent", "spam"),
    ("win money win money now", "spam"),
    ("meeting moved to today at three", "ham"),
    ("can you send the report today", "ham"),
    ("lunch meeting tomorrow works for me", "ham"),
    ("please review the report before the meeting", "ham"),
    ("your flight is confirmed for tomorrow", "ham"),
    ("send me the meeting notes please", "ham"),
]
_texts = [m[0] for m in _msgs]
_labels = [m[1] for m in _msgs]

print("Training set:")
display(pd.DataFrame(_msgs, columns=["message", "label"]))

_vec = CountVectorizer()
_Xc = _vec.fit_transform(_texts)
_vocab = _vec.get_feature_names_out()

_spam_model = MultinomialNB(alpha=1.0)
_spam_model.fit(_Xc, _labels)

_classes = list(_spam_model.classes_)
_spam_idx = _classes.index("spam")
_ham_idx = _classes.index("ham")
_word_probs = np.exp(_spam_model.feature_log_prob_)
_ratio = _word_probs[_spam_idx] / _word_probs[_ham_idx]

_evidence = pd.DataFrame({
    "word": _vocab,
    "P(word|spam)": _word_probs[_spam_idx],
    "P(word|ham)": _word_probs[_ham_idx],
    "spam/ham ratio": _ratio,
})
print(f"\nVocabulary size: {len(_vocab)}")
print("\nTop 5 most spam-indicative words:")
display(_evidence.sort_values("spam/ham ratio", ascending=False).head(5).round(3))
print("Top 5 most ham-indicative words:")
display(_evidence.sort_values("spam/ham ratio", ascending=True).head(5).round(3))

_test_msgs = [
    "click here to claim your free prize",
    "can you send the report before lunch",
    "free report today",
]
_test_X = _vec.transform(_test_msgs)
_test_pred = _spam_model.predict(_test_X)
_test_proba = _spam_model.predict_proba(_test_X)[:, _spam_idx]
print("\nPredictions on new messages:")
for _m, _p, _prob in zip(_test_msgs, _test_pred, _test_proba):
    print(f"  '{_m}' -> {_p} (P(spam) = {_prob:.3f})")

---
## What the word evidence shows

The model learned all of this from just 12 examples. `free`, `click`, and `money` are each about 5.6 times more likely to appear in a spam message than a legitimate one; `meeting` runs about 4.4 times in the other direction. Nothing about this is hidden — the entire model is a table of word probabilities you can read directly, exactly the interpretability claim made earlier in this notebook.

The borderline case is the instructive one. `free report today` mixes one strong spam word (`free`) with neutral/ham-leaning words and comes out at 61.4% spam — barely over the line, not a confident call either way. Real spam filters spend most of their effort on messages like this one, not on the obvious cases.

One honest caveat: 12 messages and 34 words is a toy example built to be readable in one sitting. Real filters train on hundreds of thousands of messages with vocabularies in the tens of thousands. The mechanism — count words, multiply likelihoods, normalize — is identical. Only the scale changes.

### Where Naive Bayes is used in practice

| Industry | Application | Why Naive Bayes |
|---|---|---|
| Email | Spam filtering | Extremely fast, works well on bag-of-words text features |
| Healthcare | Medical diagnosis (screening) | Transparent probabilistic reasoning, fast inference |
| News | Topic classification | Scales to millions of articles, high-dimensional word features |
| Finance | News sentiment for trading signals | Real-time processing requires minimal inference latency |
| Cybersecurity | Intrusion detection (signature-based) | High-frequency event streams require fast classification |

> **Naive Bayes applies Bayes theorem with a conditional independence assumption — the assumption is almost always wrong, but the model is fast, interpretable, and surprisingly competitive, especially for text classification.**

---
*Next up: 13 — Ensemble Methods, stepping back to understand why combining multiple models almost always beats any single one*